In [ ]:
import subprocess
import os
import pandas as pd
import numpy as np
from core.reader import read_ansys_csv, load_experimental_data

ANSYS_EXE_PATH = r"D:\Program Files\ANSYS Inc\ANSYS Student\v252\ansys\bin\winx64\MAPDL.exe" 
WORKING_DIR = os.getcwd()

def run_ansys_simulation(params):
    with open('chab_params.txt', 'w') as f:
        for p in params:
            f.write(f"{p}\n")

    input_file = "chab.mac"
    output_file = "ansys.out"
    
    cmd = [
        ANSYS_EXE_PATH, 
        "-b",
        "-j", "opt_run",
        "-dir", WORKING_DIR, 
        "-i", input_file, 
        "-o", output_file
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("Ошибка ANSYS:", e)
        return None

    try:
        df_res = read_ansys_csv("chab.csv")
        return df_res
    except Exception as e:
        print(f"Ошибка чтения CSV: {e}")
        return None

In [ ]:
real_experiment_data_folder = "."
df_exp = load_experimental_data(real_experiment_data_folder)

zero_row = pd.DataFrame(0.0, columns=df_exp.columns, index=[0])
zero_row['Time'] = 1.0
df_exp = pd.concat([zero_row, df_exp]).reset_index(drop=True)

def objective_function(params):
    """
    Считает ошибку между экспериментом и моделью Шабоша.
    params: [sig_y, c1, g1, c2, g2, c3, g3]
    """
    print(f"Simulating: {params}")
    
    df_ansys = run_ansys_simulation(params)
    
    if df_ansys is None or len(df_ansys) != len(df_exp):
        return 1e9

    mse_zz = np.mean((df_ansys['S_ZZ'] - df_exp['S_ZZ'])**2)
    mse_tt = np.mean((df_ansys['S_TT'] - df_exp['S_TT'])**2)
    mse_tz = np.mean((df_ansys['S_TZ'] - df_exp['S_TZ'])**2)
    
    total_error = mse_zz + mse_tt + mse_tz
    print(f"Error: {total_error:.2f}")
    return total_error

In [ ]:
import psutil

def kill_ansys_processes():
    for proc in psutil.process_iter():
        if proc.name() in ['ANSYS.exe', 'MAPDL.exe', 'ansys.exe']:
            proc.kill()

kill_ansys_processes()

In [ ]:
from scipy.optimize import differential_evolution

# Границы поиска для каждого параметра
# [sig_y, c1, g1, c2, g2, c3, g3]
bounds = [
    (200, 400),      # Sig_Y
    (1e4, 5e5),      # C1 (Жесткая кинематика)
    (100, 5000),     # gamma1 (Быстрое насыщение)
    (1e3, 5e4),      # C2
    (10, 500),       # gamma2
    (100, 1e4),      # C3
    (0, 100)         # gamma3 (Линейная часть)
]

result = differential_evolution(
    objective_function, 
    bounds, 
    strategy='best1bin', 
    maxiter=15,      # Количество поколений (увеличьте до 20-50 для точности)
    popsize=10,       # Размер популяции (увеличьте до 10-15)
    disp=True,
    polish=True,
    workers=1
)

print("\n" + "="*50)
print("ОПТИМАЛЬНЫЕ ПАРАМЕТРЫ НАЙДЕНЫ:")
print(result.x)
print("="*50)